# Proof of Concept — Notebook A (Instructor + Team Red)

Starts the competition (instructor) and plays as `team_red`. Run Notebook B
(Team Blue) in a separate kernel pointing at the SAME `SERVER_IP`/`SERVER_PORT`.

In [12]:
from pynqsim import SimulationClient

SERVER_IP = "172.20.166.171"   # <-- your server IP
SERVER_PORT = 9003             # <-- your server port

red = SimulationClient(SERVER_IP, SERVER_PORT)
print("Team Red connected to server!")

Team Red connected to server!


## Instructor: start the competition
If you get `Competition already active`, stop it first with
`red.admin_stop_competition(password=ADMIN_PASSWORD)`.

In [13]:
ADMIN_PASSWORD = "admin123"

# grid[row][col]  (row 0..5 down the list, col 0..4 across each inner list)
CARD_LAYOUT = {
    "grid": [
        # col:  0             1               2         3               4
        ["sheep",        "dining table", "person", "bicycle",      "boat"],       # row 0
        ["sofa",         "bottle",       "sofa",   "sheep",        "cat"],        # row 1
        ["car",          "bird",         "dog",    "dining table", "bottle"],     # row 2
        ["chair",        "aeroplane",    "boat",   "cow",          "person"],     # row 3
        ["cat",          "potted plant", "potted plant", "car",    "aeroplane"],  # row 4
        ["bicycle",      "cow",          "chair",  "bird",         "dog"],        # row 5
    ]
}

# Quick sanity check before sending: 30 cells, each label exactly twice.
_flat = [name for row in CARD_LAYOUT["grid"] for name in row]
from collections import Counter
_counts = Counter(_flat)
print(f"{len(_flat)} cells, {len(_counts)} unique labels")
_bad = {k: v for k, v in _counts.items() if v != 2}
if _bad:
    print("WARNING: these labels are not used exactly twice:", _bad)

red.admin_start_competition(
    "competition_card_flip",
    password=ADMIN_PASSWORD,
    card_layout=CARD_LAYOUT,
)
print("Competition started with custom layout!")

30 cells, 15 unique labels
Competition started with custom layout!


In [14]:
red.join_competition(team_id="team_red")
print("Joined as team_red!")

Joined as team_red!


In [15]:
state = red.get_competition_state()
print(f"Current turn: {state.get('current_turn')}")
print(f"Scores: {state.get('scores', {})}")

Current turn: team_red
Scores: {'team_red': 0, 'team_blue': 0}


## Color names (6x5 grid → 15 colors)
These MUST match the order the scene assigns `color_idx` in
`competition_card_flip.py` (indices 0..14).

In [5]:
# 15 colors, in the exact order the scene assigns color_idx (0..14).
COLOR_NAMES = [
    "RED", "GRN", "BLU", "YEL", "ORG",
    "PUR", "BRN", "PNK", "CYN", "MAG",
    "LIM", "TEA", "GRY", "MAR", "NVY",
]

def color_name(ci):
    return COLOR_NAMES[ci] if (ci is not None and ci < len(COLOR_NAMES)) else str(ci)

## Normal play (`flip`)
Drives the RED arm with full memory-game scoring: a **match** lets the same team
go again; a **mismatch** re-covers the cards and passes the turn.

In [32]:
def flip(client, row, col):
    """Flip a card WITH memory-game scoring/turn messaging."""
    result = client.flip_card(row, col)
    if result.get("already_flipped"):
        print(f"Card ({row},{col}) already flipped")
        return result
    print(f">>> ({row},{col}) revealed: {color_name(result['color_idx'])}")
    mr = result.get("match_result")
    if mr:
        if mr.get("matched"):
            print("*** MATCH! +1 — same team goes again! ***")
        elif mr.get("turn_ended"):
            print("No match. Cards re-covered. Turn passes to the other team.")
    return result


flip(red, 5, 2)# first card
flip(red, 5,1)#  

>>> (5,2) revealed: TEA
>>> (5,1) revealed: MAR
No match. Cards re-covered. Turn passes to the other team.


{'color_idx': 13,
 'already_flipped': False,
 'matched': False,
 'match_result': {'cards_flipped': 2,
  'matched': False,
  'turn_ended': True,
  'same_team_again': False},
 'status': 'ok'}

## `flip_raw()` — grab a cover with NO scoring / turn / re-cover
This calls the **`flip_raw` server action** (add the server snippet first). Unlike
`flip_card`, it does **not** run `record_flip`, so there's no scorekeeping, no turn
check, and mismatched covers are **not** teleported back — the cover just goes to
the discard pile and stays there. That's what lets a single client sweep the whole
board in one loop.

`flip_raw` takes an explicit `robot_id` because one arm can't reach all 30 cards —
use **red (0)** for the left columns and **blue (1)** for the right columns.

In [ ]:
def flip_raw(client, row, col, robot_id=None):
    """Grab one cover: no scoring, no turn logic, no re-cover."""
    params = {"row": row, "col": col}
    if robot_id is not None:
        params["robot_id"] = robot_id
    result = client._request("flip_raw", params)
    if result.get("already_flipped"):
        print(f"({row},{col}) already flipped")
    else:
        print(f"grabbed ({row},{col}) -> {color_name(result.get('color_idx'))}")
    return result

flip_raw(red,0,0)
flip_raw(red,0,1)
flip_raw(red,0,2)
flip_raw(red,0,3)
flip_raw(red,0,4)

## Loop: clear the whole board
Sweeps every cover off the grid. The RED arm (0) takes the left columns and the
BLUE arm (1) takes the right columns, so both stay within reach. Reads the true
grid size from the server so it adapts if the layout changes.

In [ ]:
def clear_board(client):
    grid = client.get_grid_state()
    n_rows = len(grid)
    n_cols = len(grid[0]) if n_rows else 0
    mid = n_cols // 2
    print(f"Clearing {n_rows}x{n_cols} board (red=left cols, blue=right cols)...")
    for r in range(n_rows):
        for c in range(n_cols):
            arm = 0 if c < mid else 1   # 0=red (left half), 1=blue (right half)
            try:
                flip_raw(client, r, c, robot_id=arm)
            except Exception as e:
                print(f"({r},{c}) failed: {e}")
    print("Board cleared.")

# clear_board(red)

## Cleanup

In [ ]:
red.leave_competition()
print("Team Red left.")
# red.admin_stop_competition(password=ADMIN_PASSWORD)